# Marketing LLM — LoRA Fine-Tune on Kaggle (P100-compatible)

Fine-tunes **Llama 3.2 3B Instruct** on marketing-specific Q&A pairs using vanilla `transformers` + `peft`.

**Why this version exists:**
Kaggle's REST API defaults to P100 GPUs (compute capability 6.0) which can't run modern bitsandbytes 4-bit kernels or Unsloth. This notebook uses fp16 + LoRA on a smaller 3B model that runs natively on P100.

**Trade-off:** 3B model is weaker than 8B but the fine-tuned version still specializes well on marketing patterns. The Groq 70B option remains for harder tasks.

**Expected runtime:** ~25-40 min on P100

In [ ]:
# Pre-flight: verify GPU + install deps
import torch
assert torch.cuda.is_available(), 'No GPU detected'
print(f'CUDA {torch.version.cuda} — torch {torch.__version__}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {p.total_memory/1e9:.1f} GB, cc {p.major}.{p.minor}')

import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Vanilla transformers + peft path — no Unsloth, no bitsandbytes 4-bit
pip('--upgrade', 'pip')
pip('--upgrade', 'transformers>=4.49.0,<4.55.0', 'peft>=0.14.0', 'accelerate>=1.2.0')
pip('--upgrade', 'trl>=0.12.0,<0.13.0', 'datasets', 'huggingface_hub')
print('✓ Install complete')

In [ ]:
# Clone repo for training data
import os
if not os.path.exists('/kaggle/working/Marketing'):
    os.system('git clone -q https://github.com/amittomar-hue/Marketing.git /kaggle/working/Marketing')

DATA_PATH = '/kaggle/working/Marketing/training/data/marketing_sft.jsonl'
import json
with open(DATA_PATH) as f:
    examples = [json.loads(l) for l in f]
print(f'Loaded {len(examples)} examples')
print(f'Sample: {examples[0]["intent"]} — {examples[0]["instruction"][:80]}...')

In [ ]:
# Load base model — Llama 3.2 3B Instruct in fp16 (P100 supports fp16, not bf16)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
# Fall back to a public Llama 3.2 3B if Meta's gated repo is restricted
FALLBACK_NAME = 'unsloth/Llama-3.2-3B-Instruct'

tokenizer = None
model = None
for name in [MODEL_NAME, FALLBACK_NAME]:
    try:
        print(f'Trying {name}...')
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype=torch.float16,
            device_map='auto',
            low_cpu_mem_usage=True,
        )
        print(f'✓ Loaded {name}')
        break
    except Exception as e:
        print(f'  Failed: {e}')
        continue

assert model is not None, 'Could not load any Llama 3.2 3B variant'

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram_gb = torch.cuda.memory_allocated()/1e9
print(f'GPU memory used: {vram_gb:.2f} GB')

In [ ]:
# Attach LoRA adapters via peft
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Format dataset with chat template
from datasets import Dataset

MAX_LEN = 1024
SYSTEM_PROMPT = 'You are Marketing LLM, an enterprise-grade marketing assistant. Be direct, specific, and data-driven. Format responses in clean markdown.'

def format_example(ex):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': ex['instruction']},
        {'role': 'assistant', 'content': ex['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = Dataset.from_list(examples)
ds = raw.map(format_example, remove_columns=raw.column_names)

def tokenize(ex):
    out = tokenizer(ex['text'], truncation=True, max_length=MAX_LEN, padding=False)
    out['labels'] = out['input_ids'].copy()
    return out

ds = ds.map(tokenize, remove_columns=['text'])
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds["train"])}, Eval: {len(ds["test"])}')

In [ ]:
# Train
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir='/kaggle/working/checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=5,
    fp16=True,           # P100 supports fp16
    bf16=False,          # P100 does NOT support bf16
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=20,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    optim='adamw_torch',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    report_to='none',
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    data_collator=collator,
)

stats = trainer.train()
print(f'\nTraining complete. Loss: {stats.training_loss:.4f}')

In [ ]:
# Save LoRA adapter locally
import gc, torch
gc.collect()
torch.cuda.empty_cache()

ADAPTER_DIR = '/kaggle/working/marketing-llm-3b-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✓ Adapter saved to {ADAPTER_DIR}')
os.system(f'du -sh {ADAPTER_DIR}')

In [ ]:
# Quick sanity test
model.eval()
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Write 3 Google Ads variants for an AI marketing platform.'},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
with torch.inference_mode():
    out = model.generate(input_ids=inputs, max_new_tokens=250, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Push to Hugging Face Hub
# Token must be set as Kaggle Secret 'HF_TOKEN' (Add-ons → Secrets)
# Or embedded inline (less secure)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login, HfApi
    login(token=hf_token)
    api = HfApi(token=hf_token)
    # User confirmed HF username is 'dmoop'
    REPO_NAME = 'dmoop/marketing-llm-3b-lora'
    try:
        api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)
        model.push_to_hub(REPO_NAME, token=hf_token)
        tokenizer.push_to_hub(REPO_NAME, token=hf_token)
        print(f'\n✓ Pushed to https://huggingface.co/{REPO_NAME}')
    except Exception as e:
        print(f'HF push failed: {e}')
        print(f'Adapter still saved locally at {ADAPTER_DIR} for manual download')
else:
    print('No HF_TOKEN secret found.')
    print(f'Adapter saved locally at {ADAPTER_DIR} — download via Kaggle UI or API')